# Run matrix — cola de experimentos

Notebook para Colab que define una lista de `ExperimentConfig` y las corre secuencialmente con `run_matrix()`: salta las que ya estan completas, reintenta las que quedaron incompletas de una corrida anterior, continua si una falla, y al final muestra un resumen de completadas/fallidas y MAPE promedio.

## Datos de entrada

Los 34 archivos (`IGAE_2.xlsx`, `Temperaturas promedio.csv`, y por cada una de las 8 regiones -- BCA, CEN, NES, NOR, NTE, OCC, ORI, PEN -- `{REGION}_long.csv`, `{REGION}_GEN.csv`, `{REGION}_IMP.csv`, `{REGION}_EXP.csv`) viven permanentemente en Google Drive, en `MyDrive/Bases de datos Tesis` (constante `DATA_DIR` mas abajo). Detalle exacto de columnas/formato: [`docs/DATOS_REQUERIDOS.md`](../docs/DATOS_REQUERIDOS.md).

## Modelos disponibles

`xgboost`, `lightgbm` (adaptado, ver [`docs/MODELOS_MIGRADOS.md`](../docs/MODELOS_MIGRADOS.md)), `lstm_direct`, `sarimax`, `fcnn`, `ensemble_stl`. Todos procesan las 8 regiones en una sola corrida.

## Setup (una sola celda): montar Drive, instalar dependencias, cargar el proyecto

In [ ]:
import os
import sys

from google.colab import drive
drive.mount("/content/drive")

# xgboost, tensorflow, statsmodels, pandas, numpy, scikit-learn ya vienen
# preinstalados en el runtime estandar de Colab; optuna y lightgbm no.
!pip install -q optuna lightgbm

REPO_URL = "https://github.com/CarlosT0503/Tesis-forecasting.git"
REPO_DIR = "/content/tesis_repo"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

SRC_DIR = os.path.join(REPO_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("\nSetup completo.")
print("SRC_DIR en sys.path:", SRC_DIR in sys.path)

In [ ]:
# Ruta donde viven permanentemente los datos de entrada en Google Drive.
DATA_DIR = "/content/drive/MyDrive/Bases de datos Tesis"

from tesis_forecast.config import ExperimentConfig
from tesis_forecast.matrix import run_matrix, resumen_dataframe

## Cola minima recomendada

Un experimento por modelo, todos con su configuracion vigente por defecto (mismos hiperparametros/ventanas/tratamiento de exogenas que se documentaron en la migracion). Es la cola minima para validar que los 6 pipelines corren de punta a punta en Colab con datos reales antes de construir matrices mas grandes (barridos de exogenas, distintos train_hours, etc.).

Deja `exogenas`/`train_hours` en `None` para usar el default vigente de cada modelo; se muestran explicitos aqui solo para que quede claro cual es la config de cada uno.

In [ ]:
configs = [
    ExperimentConfig(
        modelo="xgboost",
        exogenas=["Temperatura", "Primarias", "Secundarias", "Terciarias", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=336,
        notas="Cola minima -- config vigente.",
    ),
    ExperimentConfig(
        modelo="lightgbm",
        exogenas=["Temperatura", "Primarias", "Secundarias", "Terciarias", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=336,
        notas="Cola minima -- adaptado desde celda 46, no extraccion exacta.",
    ),
    ExperimentConfig(
        modelo="lstm_direct",
        exogenas=["Temperatura", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=2160,
        notas="Cola minima -- config vigente.",
    ),
    ExperimentConfig(
        modelo="sarimax",
        exogenas=["Temperatura", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=1440,
        notas="Cola minima -- config vigente, orden SARIMAX fijo (sin tuning).",
    ),
    ExperimentConfig(
        modelo="fcnn",
        exogenas=["Temperatura", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=3600,
        notas="Cola minima -- config vigente, produce 2 modelos por region (directa + STL-residuos).",
    ),
    ExperimentConfig(
        modelo="ensemble_stl",
        exogenas=["Temperatura", "IGAE", "Generacion", "Importacion", "Exportacion"],
        train_hours=3600,
        notas="Cola minima -- config vigente, es el pipeline mas pesado (2 redes + barrido AR por region).",
    ),
]

len(configs)

## Lanzar la cola

Corre secuencialmente. Puede tardar horas (Ensemble y FCNN entrenan redes por cada una de las 8 regiones). Si la sesion de Colab se desconecta a la mitad, vuelve a correr esta misma celda: `run_matrix` detecta las carpetas ya completas y las salta, y reintenta automaticamente las que quedaron incompletas.

In [ ]:
resultados = run_matrix(configs, data_dir=DATA_DIR, mount_drive=True)

resumen_dataframe(resultados)